<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>

 <p><font size="5" color='grey'> <b> LCEL Vertiefung &mdash; Brücke zu LangGraph </b></font> </br></p>

---

**Beitrag zum Leitprojekt:** LCEL ist keine eigene Agenten-Architektur, sondern eine **kontrollierte Subroutine** innerhalb des Meeting- & Research-Briefing-Agent — ein fest verdrahteter, testbarer Teilschritt (z. B. Antwort formulieren, Risiko prüfen) ohne eigene Entscheidungslogik. Sobald der Agent selbst verzweigen, sich State merken oder wiederholen muss, übernimmt LangGraph (M07) diese Rolle.

In [35]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
# Wichtig: LangSmith-Account und API-Key im EU-Workspace anlegen
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "M06-LCEL-Chains"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS

✓ OPENAI_API_KEY erfolgreich gesetzt
✓ LANGSMITH_API_KEY erfolgreich gesetzt

Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.2.15
langchain-chroma                         1.1.0
langchain-classic                        1.0.4
langchain-community                      0.4.1
langchain-core                           1.3.1
langchain-ollama                         1.1.0
langchain-openai                         1.2.0
langchain-text-splitters                 1.1.2
langgraph                                1.1.9
langgraph-checkpoint                     4.0.2
langgraph-prebuilt                       1.0.10
langgraph-sdk                            0.3.13

IP-Adresse: 34.138.185.229
Hostname: 229.185.138.34.bc.googleusercontent.com
Stadt: North Charleston
Region: South Carolina
Land: US
Koordinaten: 32.8546,-79.9748
Provider: AS396982 Google LLC
Postleitzahl: 29415
Zeitzone: America/New

# 1 | Übersicht
---

In [ ]:
from genai_lib.utilities import mermaid

# LCEL Pipe-Operator
diagram = """
%%{init: {'theme':'forest'}}%%
flowchart LR
    INPUT([Input]) --> PROMPT[ChatPromptTemplate]
    PROMPT --> LLM[LLM]
    LLM --> PARSER[OutputParser]
    PARSER --> OUT([Output])
    style INPUT fill:#4CAF50,color:#fff
    style OUT fill:#2196F3,color:#fff
"""
mermaid(diagram, width=700)

Dieses Modul vertieft LCEL mit dem Pipe-Operator `|`.

Im roten Faden übernimmt LCEL eine Zwischenrolle: einfache Briefing-Agent-Abläufe lassen sich als lineare Ketten modellieren. Sobald Zustände, Wiederholungen oder Routing nötig werden, führt derselbe Gedanke zu LangGraph.


In [ ]:
# Imports für LCEL-Beispiele
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

llm = init_chat_model(BASELINE)
parser = StrOutputParser()


# 2 | Was ist LCEL?
---

> **Grundlagen** (Pipe-Operator, erste Chain) wurden in **M02 Kap. 3** eingeführt. Hier folgt die Vertiefung.

LCEL (LangChain Expression Language) beschreibt Pipelines aus **Runnable**-Bausteinen mit klarem Input/Output-Protokoll:

- Jeder Baustein implementiert `.invoke()`, `.stream()`, `.batch()`
- Bausteine sind kombinierbar: `a | b | c` -> neue Runnable
- Das Protokoll gilt für: Prompts, LLMs, Parser, Tools, eigene Funktionen (`RunnableLambda`)

Für den Meeting-Briefing-Agent ist LCEL gut geeignet, solange der Ablauf linear bleibt: Frage normalisieren -> Antwort entwerfen -> Text nachbearbeiten. Sobald Verzweigungen, Wiederholungen oder menschliche Freigaben nötig werden, ist LangGraph die bessere Wahl.


In [37]:
# Mini-Check: Ein einfacher RunnableLambda-Schritt
normalize = RunnableLambda(lambda x: x.strip().lower())
print(normalize.invoke("   RAG Evaluation   "))


rag evaluation


# 3 | Pipe-Operator
---

Mit `|` werden Schritte zu einer Kette verbunden. Dadurch wird der Datenfluss explizit und gut testbar.


In [ ]:
research_prompt = ChatPromptTemplate.from_template(
    "Erkläre den Begriff {thema} im Kontext eines Meeting-Briefing-Agenten in zwei Sätzen."
)

chain = research_prompt | llm | parser
antwort = chain.invoke({"thema": "Retrieval Augmented Generation"})
mprint(antwort)


# 4 | Minimalbeispiel: prompt | llm
---

Ohne Parser entsteht ein AIMessage-Objekt. Mit `StrOutputParser()` kann direkt mit Text weitergearbeitet werden.


In [39]:
raw_chain = research_prompt | llm
parsed_chain = research_prompt | llm | parser

raw = raw_chain.invoke({"thema": "Korpusabdeckung"})
parsed = parsed_chain.invoke({"thema": "Korpusabdeckung"})

print(type(raw).__name__)
print(type(parsed).__name__)


AIMessage
TextAccessor


# 5 | Komplexe Chains
---

`RunnableParallel` erlaubt parallele Teilketten.
So lassen sich zum Beispiel eine kurze Antwort und eine kritische Quellenprüfung gleichzeitig erzeugen.


In [ ]:
base_prompt = ChatPromptTemplate.from_template(
    "Analysiere die folgende Briefing-Frage: {text}"
)

summary_chain = (
    base_prompt
    | llm
    | parser
    | RunnableLambda(lambda t: "Zusammenfassung:\n" + t)
)

critic_chain = (
    ChatPromptTemplate.from_template(
        "Nenne 3 mögliche Quellen- oder Qualitätsrisiken in dieser Briefing-Frage:\n{text}"
    )
    | llm
    | parser
    | RunnableLambda(lambda t: "Prüfung:\n" + t)
)

combined_chain = RunnableParallel(
    summary=summary_chain,
    critique=critic_chain,
)

text = "Warum verbessert RAG die Zuverlässigkeit eines Meeting-Briefing-Agenten?"
out = combined_chain.invoke({"text": text})

mprint('### Ergebnis')
mprint('---')
mprint(out["summary"])
mprint('---')
mprint(out["critique"])


# 6 | Chain-Debugging mit LangSmith
---

Mit LangSmith wird sichtbar:
- welche Chain-Schritte aufgerufen wurden
- welche Inputs und Outputs pro Schritt entstanden sind
- wo Fehler oder unerwartete Antworten auftreten


run_cfg = {
    "run_name": "M06_Kap6_ResearchChainTrace",
    "tags": ["M06", "lcel", "meeting-research-briefing"]
}

traced = combined_chain.with_config(**run_cfg)
trace_output = traced.invoke({
    "text": "Wie sollte ein Meeting-Briefing-Agent Antworten zu RAG belegen?"
})

mprint("## Trace-Ausgabe")
mprint(trace_output["summary"][:500])


<p><font color='black' size="5">
Tracing
</font></p>

In [41]:
# Beispiel-Komponenten (falls noch nicht definiert)
prompt = ChatPromptTemplate.from_template("Erzähle mir kurz etwas über {topic}")

# 1. Tracing-Konfiguration vorab festlegen
run_cfg = {
    "run_name": "M06_Kap6_Chain_Debugging",
    "tags":     ["M06", "lcel", "chain", "debug"]
}

# 2. with_config() anwenden, dann invoke()
debug_chain = (prompt | llm | parser).with_config(**run_cfg)
dbg = debug_chain.invoke({"topic": "Wann lohnt sich Caching?"})

mprint("### Trace erstellt")
mprint('---')
mprint(dbg)

### Trace erstellt

---

Caching lohnt sich in verschiedenen Szenarien, insbesondere wenn es darum geht, die Leistung und Effizienz von Anwendungen oder Systemen zu verbessern. Hier sind einige Situationen, in denen Caching besonders vorteilhaft ist:

1. **Häufige Datenzugriffe**: Wenn bestimmte Daten häufig abgerufen werden, kann Caching die Zugriffszeiten erheblich verkürzen, da die Daten nicht jedes Mal neu geladen oder berechnet werden müssen.

2. **Teure Berechnungen**: Wenn die Generierung von Daten (z. B. durch komplexe Berechnungen oder Datenbankabfragen) ressourcenintensiv ist, kann das Zwischenspeichern der Ergebnisse die Systemlast reduzieren und die Reaktionszeiten verbessern.

3. **Statische Inhalte**: Bei Webanwendungen, die viele statische Inhalte (wie Bilder, CSS oder JavaScript-Dateien) bereitstellen, kann Caching die Ladezeiten für Benutzer erheblich verkürzen.

4. **Skalierbarkeit**: In verteilten Systemen oder bei hohem Benutzeraufkommen kann Caching helfen, die Last auf Backend-Servern zu verringern und die Skalierbarkeit zu verbessern.

5. **Datenkonsistenz**: In Szenarien, in denen Daten nicht häufig aktualisiert werden, kann Caching eine effiziente Möglichkeit sein, um die Leistung zu steigern, ohne dass die Konsistenz der Daten beeinträchtigt wird.

Insgesamt ist Caching eine effektive Strategie, um die Leistung zu optimieren, die Benutzererfahrung zu verbessern und die Ressourcennutzung zu minimieren. Es ist jedoch wichtig, die Cache-Strategie sorgfältig zu planen, um Probleme wie veraltete Daten zu vermeiden.

# 7 | Streaming mit LCEL

- Unterschied `invoke()` vs. `stream()`/`astream()`
- Wann Streaming UX verbessert: lange Research-Antworten, Live-Feedback, schrittweise Sichtbarkeit
- Mini-Demo: gleiche Chain einmal normal, einmal gestreamt

---


In [42]:
stream_prompt = ChatPromptTemplate([
    ("system", "Rolle: Meeting-Briefing-Agent für Retrieval Augmented Generation (RAG). RAG immer als Retrieval Augmented Generation verstehen."),
    ("user", "Formuliere eine kurze Briefing-Agent-Antwort zu: {frage}"),
])
stream_chain = stream_prompt | llm | parser

normal = stream_chain.invoke({"frage": "Warum braucht Retrieval Augmented Generation (RAG) Quellenbindung?"})
print(normal[:300])


Retrieval Augmented Generation (RAG) benötigt Quellenbindung, um die Glaubwürdigkeit und Nachvollziehbarkeit der generierten Informationen zu gewährleisten. Durch die Verknüpfung von generierten Inhalten mit spezifischen Quellen können Nutzer die Herkunft der Informationen überprüfen, was das Vertra


In [43]:
chunks = []
for chunk in stream_chain.stream({"frage": "Warum braucht Retrieval Augmented Generation (RAG) Quellenbindung?"}):
    chunks.append(chunk)
    print(chunk, end="")

stream_text = "".join(chunks)


Retrieval Augmented Generation (RAG) benötigt Quellenbindung, um die Nachvollziehbarkeit und Vertrauenswürdigkeit der generierten Informationen zu gewährleisten. Durch die Verknüpfung von generierten Inhalten mit spezifischen Quellen können Nutzer die Herkunft der Informationen überprüfen und deren Genauigkeit beurteilen. Dies ist besonders wichtig in Kontexten, in denen präzise und verlässliche Daten entscheidend sind, wie in der Wissenschaft, im Journalismus oder in der medizinischen Beratung. Zudem fördert die Quellenbindung die Transparenz des Modells und ermöglicht eine bessere Fehleranalyse, da Nutzer nachvollziehen können, auf welche Informationen das Modell zurückgegriffen hat.

In [44]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M06-LCEL-Chains", limit=3, show_steps=True)

## LangSmith Trace — `M06-LCEL-Chains`

| Run | Status | Dauer | Child-Runs |
|-----|--------|-------|------------|
| `RunnableSequence` | ✅ success | 2.3s | 0 |
| `RunnableSequence` | ✅ success | 2.8s | 0 |
| `M06_Kap6_Chain_Debugging` | ✅ success | 11.1s | 0 |


### Steps — letzter Run: `RunnableSequence`

| # | Typ | Name | Status | Dauer |
|---|-----|------|--------|-------|
| 1 | `parser` | `StrOutputParser` | ✅ | 2.1s |
| 2 | `llm` | `ChatOpenAI` | ✅ | 2.3s |
| 3 | `prompt` | `ChatPromptTemplate` | ✅ | 0.0s |

# 8 | Ausblick: Brücke zu LangGraph
---

LCEL-Chains sind linear: Jeder Schritt folgt dem nächsten. **LangGraph** löst die zentrale Einschränkung von Chains:
**keine Schleifen, keine Verzweigungen, kein persistenter State**.

| LCEL-Konzept | LangGraph-Äquivalent | Unterschied |
|---|---|---|
| `prompt | llm | parser` | Sequenz von Nodes | LangGraph kann zurückspringen |
| `RunnableParallel` | Parallele Edges | LangGraph steuert den Fluss dynamisch |
| `RunnableLambda` | Node-Funktion | Nodes haben Zugriff auf den gesamten State |
| `chain.invoke()` | `graph.invoke()` | Identische API |

```
LCEL:       A -> B -> C -> Ende          (immer linear)
LangGraph:  A -> B -> C -> A            (Schleife)
            A -> B oder D               (Verzweigung)
```

**Übergangscheck M06 -> M07**

| Frage | Wenn ja, reicht LCEL? | Wenn nein, nächstes Modul |
|---|---:|---|
| Ist der Ablauf immer gleich? | ja | M07: Routing mit State |
| Braucht der Agent eine Entscheidung über den nächsten Schritt? | nein | M07: Conditional Edges |
| Muss ein Schritt wiederholt oder abgebrochen werden? | nein | M09/M10: Schleifen und Tool-Loops |
| Muss Kontext über mehrere Schritte erhalten bleiben? | nur begrenzt | M16/M18: Checkpointing und Memory |

> Alles, was über Chains gelernt wurde, gilt in LangGraph weiter. Der Unterschied: Nodes können auf State zugreifen und der Graph kann den nächsten Schritt steuern. Genau deshalb folgt jetzt **M07**.


# A | Aufgaben
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen unten bieten Anregungen; alternative Herausforderungen sind möglich.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Geeignet ist sie zum Beispiel, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


**Grundlagen**
- Eine einfache LCEL-Chain mit Prompt -> LLM -> Parser bauen.
- Die Chain mit einer Briefing-Agent-Frage testen.

**✅ Erledigt wenn:** `meine_chain.invoke({'thema': 'RAG'})` gibt einen String zurück und enthält mindestens 3 Schritte.


In [ ]:
# Grundlagen: Einfache LCEL-Chain (Prompt | LLM | Parser)
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. ChatPromptTemplate mit system/user-Rollen definieren (Rolle: Meeting-Briefing-Agent)
# 2. llm = init_chat_model(BASELINE)
# 3. meine_chain = prompt | llm | StrOutputParser(), mit einer Briefing-Frage testen

**Aufbau**
1. Eine LCEL-Chain mit mindestens 4 Schritten bauen: Prompt -> LLM -> Parser -> Postprocessing.
2. Eine kleine Bewertungsfunktion (`RunnableLambda`) implementieren, die Antwortlänge oder Quellenhinweis prüft.
3. Die Chain mit 3 unterschiedlichen Briefing-Fragen testen und Auffälligkeiten notieren.

**✅ Erledigt wenn:** `meine_aufbau_chain` enthält mindestens 4 Schritte inkl. `RunnableLambda`; drei verschiedene Inputs geben sinnvolle Ausgaben.


In [ ]:
# Aufbau: 4-Schritt-Chain mit RunnableLambda
from langchain_core.runnables import RunnableLambda

# 1. nachbearbeitung(text: str) -> str: prüft z.B. ob ein Quellenhinweis vorkommt
# 2. meine_aufbau_chain = prompt | llm | StrOutputParser() | RunnableLambda(nachbearbeitung)
# 3. Mit mindestens drei unterschiedlichen Themen testen

**Vertiefung**
- Eine zweite Teilkette ergänzen und beide mit `RunnableParallel` kombinieren.
- Eine Teilkette formuliert eine kurze Antwort, die andere prüft Quellen- oder Risikohinweise.

**✅ Erledigt wenn:** `meine_parallel_chain` enthält einen `RunnableParallel` mit mindestens 2 Teilketten; LangSmith ist aktiv.


In [ ]:
# Vertiefung: RunnableParallel mit 2 Teilketten
from langchain_core.runnables import RunnableParallel

# 1. Zwei Teilketten definieren: eine Antwort-Kette, eine Risiko-/Quellenprüf-Kette
# 2. meine_parallel_chain = RunnableParallel(antwort=..., risiko=...)
# 3. Mit einer Briefing-Frage testen (LangSmith-Tracing ist bereits aktiv)

<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [KI-Agenten-Architektur](https://editor.p5js.org/ralf.bendig.rb/full/Viso2emNI)
- [LangGraph](https://editor.p5js.org/ralf.bendig.rb/full/EUzaFq4C4)


# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [LangChain](https://ralf-42.github.io/Agenten/05-frameworks/langchain.html)
- [LangChain Best Practices](https://ralf-42.github.io/Agenten/05-frameworks/langchain-best-practices.html)
- [LangChain/LangGraph Cheatsheet](https://ralf-42.github.io/Agenten/05-frameworks/langchain-langgraph-cheatsheet.html)
- [State Management](https://ralf-42.github.io/Agenten/04-agenten-implementierung/ablauf-zustand/state-management.html)
- [Einsteiger LangGraph](https://ralf-42.github.io/Agenten/05-frameworks/einsteiger-langgraph.html)
